In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog workspace")
spark.sql("create schema if not exists silver_schema")

silver_run_id= str(uuid.uuid4())
print("Current silver run id ",silver_run_id)

# STEP 2:  Creating silver control table 

will store silver processing state from each entity.

will track:

-latest bronze run already processed by silver

-latest bronze ingestion time that has already been processed by silver

-how many rows were merged in latest silver run

In [0]:
spark.sql("""
          create table if not exists workspace.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              row_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

# Step 3 : Helper Functions

-contains reuseable logic
> upsert_to_siver(): merges clean/transformed rows into Silver target table.

>get_last_processed_bronze_ingested_at():reads the silver watermark

>upsert_silver_control():updates silver control table

>get_incremental_bronze(): reads on;y new brnze rows silver has not pcessed yet.

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        (
        dt.alias("target").merge(
            df_source.alias("source"),
            f"target.{join_key} = source.{join_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl=(
        spark.table("workspace.silver_schema.processing_control")
              .filter(
                  (F.col("layer")=="silver") &
                  (F.col("entity_name")==entity_name) &
                  (F.col("run_status")=="SUCCESS")
              ).orderBy(F.col("updated_at").desc()).limit(1))
    rows=ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_bronze_ingested_at"]
    

In [0]:
def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    row_merged,
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(row_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            row_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """,
    )
    dt = DeltaTable.forName(spark, "workspace.silver_schema.processing_control")
    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"), "t.entity_name = s.entity_name and t.layer = s.layer"
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
                "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
                "row_merged": "s.row_merged",
                "run_status": "s.run_status",
                "silver_run_id": "s.silver_run_id",
                "updated_at": "s.updated_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    return bronze_df.filter(
        F.col("bronze_ingested_at")>F.lit(last_ingested_at)
    ),last_ingested_at


# Step 4 Orders incremental processing

In [0]:
df_raw=spark.sql("select * from workspace.bronze_schema.orders_raw")
display(df_raw)

In [0]:
#step 4- orders incremental processing
# read only the Bronze order rows that silver has not processed

orders_inc,last_order_injested_at=get_incremental_bronze("workspace.bronze_schema.orders_raw","orders")
#count it
orders_inc_count=orders_inc.count()
print(f"orders rows_to_process_in_silver= {orders_inc_count}")

if orders_inc_count> 0:
    #creating the window to keep the latest order record for each order id.
    order_window=Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    #starting the Silver order-cleaning pipeline for standarizing and dedeuplicating raw order records.

    orders_cleaned=(
        orders_inc
        .withColumn("order_status",F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status",F.when(F.col("order_status")=="",F.lit(None)).otherwise(F.col("order_status")))

        #removing formatting character from order_amount so that it could be casted into numeric type
        .withColumn("order_amount",F.regexp_replace(F.col("order_amount"),r"[$, ]",""))
        .withColumn("order_amount",F.when(F.trim(F.col("order_amount")).isin("N/A","NULL","??",""),None).otherwise(F.col("order_amount")))
        .withColumn("order_amount",F.col("order_amount").cast("double"))
        .withColumn("created_at",F.to_timestamp("created_at"))
        .withColumn("updated_at",F.to_timestamp("updated_at"))

        #assigning a row inside each business key so that we can keep only the latest version of that record
        .withColumn("row_rank",F.row_number().over(order_window))
        #keeping only the latest record
        .filter(F.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run_id))

    )
    #Merging the cleaned or validated Silver dataset into delta target table.
    upsert_to_silver(orders_cleaned,
        "workspace.silver_schema.orders_cleaned",
        "order_id"
        )
    # Apply Silver data-quality rules to the cleaned order records.
    orders_validated = (
    orders_cleaned
    .withColumn(
        "to_be_verified_by_orders_team",
        F.when(F.col("customer_id").isNull(), "verify_customer_id")
        .when(F.col("product_id").isNull(), "verify_product_id")
        .when(
            F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""),
            "verify_order_status"
        )
        .when(
            F.col("order_amount").isNull() | (F.col("order_amount") <= 0),
            "verify_order_amount"
        )
        .otherwise("No Issues")
    )
    .withColumn(
        "check_order_amount",
        F.when(
            F.col("order_amount").isNull() | (F.col("order_amount") <= 0),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn("order_date", F.to_date("created_at"))
    .withColumn("order_year", F.year("created_at"))
    .withColumn("order_month", F.month("created_at"))
    .withColumn("order_day", F.dayofmonth("created_at"))
    .withColumn("order_dow", F.date_format("created_at", "E"))
)

    # Keep only valid order rows for the transformed Silver table.
    orders_good = orders_validated.filter(
        F.col("to_be_verified_by_orders_team") == "No Issues"
    )
    # Send invalid order rows to the quarantine dataset for manual review.
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "workspace.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad order rows to the quarantine table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "workspace.silver_schema.orders_quarantine"
    )

    # Get latest ingestion timestamp from processed batch
    mx_ingested = orders_inc.agg(
        F.max("bronze_ingested_at").alias("mx")
    ).collect()[0]["mx"]

    # Get latest run_id for that ingestion timestamp
    mx_run = (
        orders_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    upsert_silver_control("orders",mx_run,mx_ingested,orders_good.count())
else:
    print("No new orders Bronze Row for Silver")
    upsert_silver_control(
        "orders",
        None,
        last_order_injested_at,
        orders_inc_count
    )

In [0]:
spark.table("workspace.silver_schema.processing_control").orderBy(F.col("updated_at").desc()).show()